# Overview features

This notebook is for testing the first feature layer for the overview page.

The previous notebook was about understanding the raw data and checking that loading and cleaning worked. Here the goal is simpler: make sure the overview metrics and summary tables look reasonable before moving into charts and Streamlit.

## Imports and setup

This notebook uses the project modules directly so the feature logic stays in Python files instead of getting buried in notebook cells.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loader import (
    load_bills,
    load_transcripts_topic,
    load_transcripts_text,
    load_voting_sessions,
)
from src.data.cleaning import (
    clean_dataframe_basic,
    filter_transcript_parties,
    parse_vote_data_column,
)
from src.features.overview_features import (
    get_overview_kpis,
    get_overview_tables,
    get_bills_by_parliament,
    get_bills_by_stage,
    get_bills_by_status,
    get_transcripts_by_parliament,
    get_transcripts_by_party,
    get_top_level_2_topics,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)

## Load and clean the datasets

For the overview page I mainly need bills, transcripts, and voting sessions.

I use the transcript file with text by default here, because it is the richer version. I also keep the topic-only transcript file loaded so I can compare the outputs later if needed.

In [2]:
bills_df = load_bills(verbose=False)
transcripts_text_df = load_transcripts_text(verbose=False)
transcripts_topic_df = load_transcripts_topic(verbose=False)
voting_df = load_voting_sessions(verbose=False)

bills_clean = clean_dataframe_basic(
    bills_df,
    list_like_columns=["topics", "level_2_topics"],
)

transcripts_text_clean = clean_dataframe_basic(
    transcripts_text_df,
    unix_time_columns=["time"],
    list_like_columns=["level_2_topics", "level_3_topics"],
)
transcripts_text_clean = filter_transcript_parties(transcripts_text_clean)

transcripts_topic_clean = clean_dataframe_basic(
    transcripts_topic_df,
    unix_time_columns=["time"],
    list_like_columns=["level_2_topics", "level_3_topics"],
)
transcripts_topic_clean = filter_transcript_parties(transcripts_topic_clean)

voting_clean = clean_dataframe_basic(
    voting_df,
    datetime_columns=["date"],
)

voting_clean = parse_vote_data_column(voting_clean)


## Quick shape check

Just a quick sanity check before generating features.

In [3]:
{
    "bills_clean": bills_clean.shape,
    "transcripts_text_clean": transcripts_text_clean.shape,
    "transcripts_topic_clean": transcripts_topic_clean.shape,
    "voting_clean": voting_clean.shape,
}

{'bills_clean': (1140, 17),
 'transcripts_text_clean': (183716, 13),
 'transcripts_topic_clean': (304800, 12),
 'voting_clean': (800, 12)}

## Overview KPIs

These are the headline numbers I want at the top of the overview page.

In [4]:
overview_kpis = get_overview_kpis(
    bills_df=bills_clean,
    transcripts_df=transcripts_text_clean,
    voting_df=voting_clean,
)

overview_kpis

{'total_bills': 1140,
 'total_transcript_rows': 183716,
 'total_voting_rows': 800,
 'unique_speakers': 404,
 'unique_sponsors': 359,
 'unique_parties': 6,
 'unique_level_2_topics': 22}

In [5]:
overview_kpis_df = pd.DataFrame(
    {
        "metric": list(overview_kpis.keys()),
        "value": list(overview_kpis.values()),
    }
)

overview_kpis_df

,metric,value
0,total_bills,1140
1,total_transcript_rows,183716
2,total_voting_rows,800
3,unique_speakers,404
4,unique_sponsors,359
5,unique_parties,6
6,unique_level_2_topics,22


## Overview tables

These grouped tables are the base for the first charts and summary sections.

In [6]:
overview_tables = get_overview_tables(
    bills_df=bills_clean,
    transcripts_df=transcripts_text_clean,
)

overview_tables.keys()

dict_keys(['bills_by_parliament', 'bills_by_stage', 'bills_by_status', 'transcripts_by_parliament', 'transcripts_by_party', 'top_bill_topics', 'top_transcript_topics'])

In [7]:
overview_tables["bills_by_parliament"]

,parliament,bill_count
0,42,441
1,43,287
2,44,412


In [8]:
overview_tables["bills_by_stage"]

,stage,bill_count
0,First reading,762
1,Royal assent,242
2,Second reading,113
3,Third reading,23


In [9]:
overview_tables["bills_by_status"]

,status,bill_count
0,Outside the Order of Precedence,409
1,Royal assent received,244
2,At second reading in the House of Commons,119
3,At second reading in the Senate,101
4,Bill defeated,100
5,At consideration in committee in the Senate,45
6,Bill not proceeded with,40
7,At report stage in the House of Commons,23
8,Senate bill awaiting first reading in the House of Commons,17
9,At third reading in the Senate,12


In [10]:
overview_tables["transcripts_by_parliament"]

,parliament,transcript_count
0,43,52818
1,44,130898


In [11]:
overview_tables["transcripts_by_party"].head(20)

,party,transcript_count
0,Liberal,76099
1,Conservative,59093
2,NDP,25574
3,Bloc Québécois,19224
4,Green Party,3272
5,Independent,454


In [12]:
overview_tables["top_bill_topics"].head(20)

,level_2_topic,count
0,Government Operations,1094
1,Law and Crime,1000
2,Health,166
3,Culture,126
4,Social Welfare,119
5,Civil Rights,104
6,Macroeconomics,91
7,Environment,74
8,Technology,63
9,Labor,54


In [13]:
overview_tables["top_transcript_topics"].head(20)

,level_2_topic,count
0,Government Operations,189918
1,Other,108245
2,International Affairs,77799
3,Health,69357
4,Macroeconomics,66175
5,Law and Crime,63844
6,Civil Rights,51286
7,Social Welfare,40815
8,Defense,37798
9,Public Lands,31763


## Compare transcript variants

Since there are two transcript versions, it is useful to compare their high-level outputs once before deciding what should feed the overview page.

In [14]:
text_kpis = get_overview_kpis(
    bills_df=bills_clean,
    transcripts_df=transcripts_text_clean,
    voting_df=voting_clean,
)

topic_only_kpis = get_overview_kpis(
    bills_df=bills_clean,
    transcripts_df=transcripts_topic_clean,
    voting_df=voting_clean,
)

transcript_comparison_df = pd.DataFrame(
    {
        "metric": [
            "total_transcript_rows",
            "unique_speakers",
            "unique_parties",
            "unique_level_2_topics",
        ],
        "with_text": [
            text_kpis["total_transcript_rows"],
            text_kpis["unique_speakers"],
            text_kpis["unique_parties"],
            text_kpis["unique_level_2_topics"],
        ],
        "topic_only": [
            topic_only_kpis["total_transcript_rows"],
            topic_only_kpis["unique_speakers"],
            topic_only_kpis["unique_parties"],
            topic_only_kpis["unique_level_2_topics"],
        ],
    }
)

transcript_comparison_df

,metric,with_text,topic_only
0,total_transcript_rows,183716,304800
1,unique_speakers,404,0
2,unique_parties,6,8
3,unique_level_2_topics,22,22


## Test individual feature functions

I also want to check the functions one by one so it is easier to spot where something looks off.

In [15]:
get_bills_by_parliament(bills_clean)

,parliament,bill_count
0,42,441
1,43,287
2,44,412


In [16]:
get_bills_by_stage(bills_clean).head(20)

,stage,bill_count
0,First reading,762
1,Royal assent,242
2,Second reading,113
3,Third reading,23


In [17]:
get_bills_by_status(bills_clean).head(20)

,status,bill_count
0,Outside the Order of Precedence,409
1,Royal assent received,244
2,At second reading in the House of Commons,119
3,At second reading in the Senate,101
4,Bill defeated,100
5,At consideration in committee in the Senate,45
6,Bill not proceeded with,40
7,At report stage in the House of Commons,23
8,Senate bill awaiting first reading in the House of Commons,17
9,At third reading in the Senate,12


In [18]:
get_transcripts_by_parliament(transcripts_text_clean)

,parliament,transcript_count
0,43,52818
1,44,130898


In [19]:
get_transcripts_by_party(transcripts_text_clean).head(20)

,party,transcript_count
0,Liberal,76099
1,Conservative,59093
2,NDP,25574
3,Bloc Québécois,19224
4,Green Party,3272
5,Independent,454


In [20]:
get_top_level_2_topics(bills_clean, top_n=20)

,level_2_topic,count
0,Government Operations,1094
1,Law and Crime,1000
2,Health,166
3,Culture,126
4,Social Welfare,119
5,Civil Rights,104
6,Macroeconomics,91
7,Environment,74
8,Technology,63
9,Labor,54


In [21]:
get_top_level_2_topics(transcripts_text_clean, top_n=20)

,level_2_topic,count
0,Government Operations,189918
1,Other,108245
2,International Affairs,77799
3,Health,69357
4,Macroeconomics,66175
5,Law and Crime,63844
6,Civil Rights,51286
7,Social Welfare,40815
8,Defense,37798
9,Public Lands,31763


## Topic parsing check

The topic columns started as list-like strings, so this is just to confirm that the parsed list columns exist and look right.

In [22]:
[col for col in bills_clean.columns if "topics" in col.lower()]

['topics', 'level_2_topics', 'topics_list', 'level_2_topics_list']

In [23]:
[col for col in transcripts_text_clean.columns if "topics" in col.lower()]

['level_3_topics',
 'level_2_topics',
 'level_2_topics_list',
 'level_3_topics_list']

In [24]:
bills_clean[["level_2_topics", "level_2_topics_list"]].head(10)

,level_2_topics,level_2_topics_list
0,"['Transportation', 'Law and Crime']","[Transportation, Law and Crime]"
1,"['Transportation', 'Government Operations', 'Government Operations']","[Transportation, Government Operations, Government Operations]"
2,"['Government Operations', 'Law and Crime', 'Public Lands']","[Government Operations, Law and Crime, Public Lands]"
3,"['Government Operations', 'Social Welfare', 'International Affairs']","[Government Operations, Social Welfare, International Affairs]"
4,"['Government Operations', 'Health', 'Law and Crime']","[Government Operations, Health, Law and Crime]"
5,"['Government Operations', 'Social Welfare', 'International Affairs']","[Government Operations, Social Welfare, International Affairs]"
6,"['Health', 'Culture', 'Civil Rights']","[Health, Culture, Civil Rights]"
7,"['Law and Crime', 'Social Welfare']","[Law and Crime, Social Welfare]"
8,"['Civil Rights', 'Law and Crime', 'Environment']","[Civil Rights, Law and Crime, Environment]"
9,"['Government Operations', 'Law and Crime', 'Macroeconomics']","[Government Operations, Law and Crime, Macroeconomics]"


In [25]:
transcripts_text_clean[["level_2_topics", "level_2_topics_list"]].head(10)

,level_2_topics,level_2_topics_list
0,"['Other', 'Social Welfare', 'Law and Crime', 'Law and Crime', 'Civil Rights']","[Other, Social Welfare, Law and Crime, Law and Crime, Civil Rights]"
1,"['Other', 'Defense', 'Government Operations', 'Defense']","[Other, Defense, Government Operations, Defense]"
2,"['Technology', 'International Affairs', 'Other', 'Law and Crime', 'International Affairs', 'Health', 'Other', 'Health', 'Civil Rights']","[Technology, International Affairs, Other, Law and Crime, International Affairs, Health, Other, Health, Civil Rights]"
3,"['Technology', 'International Affairs', 'Other', 'Law and Crime', 'International Affairs', 'Health', 'Other', 'Health', 'Civil Rights']","[Technology, International Affairs, Other, Law and Crime, International Affairs, Health, Other, Health, Civil Rights]"
4,"['Public Lands', 'Government Operations', 'Education', 'Education', 'Energy', 'Social Welfare', 'International Affairs', 'Law and Crime', 'Health']","[Public Lands, Government Operations, Education, Education, Energy, Social Welfare, International Affairs, Law and Crime, Health]"
5,"['Other', 'Other', 'Civil Rights', 'Social Welfare']","[Other, Other, Civil Rights, Social Welfare]"
6,"['International Affairs', 'Other', 'Government Operations', 'Health']","[International Affairs, Other, Government Operations, Health]"
7,"['International Affairs', 'Other', 'Government Operations', 'Law and Crime', 'Other', 'Civil Rights']","[International Affairs, Other, Government Operations, Law and Crime, Other, Civil Rights]"
8,"['International Affairs', 'Civil Rights']","[International Affairs, Civil Rights]"
9,"['Health', 'Other', 'Civil Rights', 'Social Welfare']","[Health, Other, Civil Rights, Social Welfare]"
